# Smart Meter KPI Dashboard

Run notebook 11 first. Then run the cells below in order.

In [ ]:
import os,html
CATALOG=os.getenv('AIDP_CATALOG','aidp_poc')
items=spark.table(f'{CATALOG}.gold.layer_kpis').collect()
class View:
 def __init__(self,x): self.x=x
 def _repr_html_(self): return self.x
 def __repr__(self): return 'HTML renderer unavailable.'
View('''<style>.title{font-family:Arial;font-size:42px;font-weight:bold;color:#fff;text-align:center;padding:25px;border-radius:18px;background:linear-gradient(120deg,#06152d,#124b90,#06152d);background-size:200% 100%;animation:flow 3s linear infinite,glow 1.4s ease-in-out infinite alternate}@keyframes flow{0%{background-position:0% 50%}100%{background-position:200% 50%}}@keyframes glow{from{box-shadow:0 0 10px #4cc9ff}to{box-shadow:0 0 30px #7be8ff}}</style><div class=title>⚡ Smart Meter Dashboard</div>''')

## Live data capture

In [ ]:
def tiles(layer,title,icon,color):
 data=[x for x in items if x.layer==layer]
 cards=''.join('<div class=tile><div class=icon>'+icon+'</div><div class=label>'+html.escape(str(x.kpi))+'</div><div class=value>'+html.escape(str(x.value))+'</div></div>' for x in data)
 return '<style>.zone{font-family:Arial;color:#fff;background:#071c3d;padding:20px;border-radius:16px}.zone h2{margin:0 0 16px;color:'+color+'}.tiles{display:flex;gap:12px;overflow-x:auto;padding-bottom:6px}.tile{min-width:190px;flex:1;background:#0d2854;border:1px solid '+color+';border-radius:13px;padding:15px;box-shadow:0 9px 20px #0007;transition:.25s}.tile:hover{transform:translateY(-6px) perspective(700px) rotateX(3deg)}.icon{font-size:23px}.label{font-size:11px;color:#d6e7ff;margin-top:8px}.value{font-size:24px;font-weight:bold;color:#fff;margin-top:6px}</style><div class=zone><h2>'+icon+' '+title+'</h2><div class=tiles>'+cards+'</div></div>'
View(tiles('BRONZE','Live data capture','📡','#7eeaff'))

## Data quality and coverage

In [ ]:
View(tiles('SILVER','Data quality and coverage','✓','#9dffbd'))

## Energy and network performance

In [ ]:
View(tiles('GOLD','Energy and network performance','⚡','#ffd07a'))

## Forecast intelligence

In [ ]:
View(tiles('ML','Forecast intelligence','◈','#c7a9ff'))

## Top 100 predicted meter consumptions

In [ ]:
preds=spark.table(f'{CATALOG}.gold.dashboard_predictions').orderBy('prediction_kwh',ascending=False).limit(100).collect()
rows=''.join(f'<tr><td>{i+1}</td><td>{html.escape(str(x.meter_id))}</td><td>{x.prediction_kwh:,.4f} kWh</td><td>{html.escape(str(x.interval_start_utc))}</td></tr>' for i,x in enumerate(preds))
View('''<style>.list{font-family:Arial;color:#fff;background:#071c3d;padding:20px;border-radius:16px}.list h2{color:#c7a9ff;margin-top:0}table{width:100%;border-collapse:collapse}th,td{padding:9px;border-bottom:1px solid #ffffff2b;text-align:left;color:#fff}td:nth-child(1),td:nth-child(3),th:nth-child(1),th:nth-child(3){text-align:right}</style><div class=list><h2>◈ Top 100 forecasted consumptions</h2><table><tr><th>Rank</th><th>Meter</th><th>Predicted consumption</th><th>Future 15-minute interval</th></tr>'''+rows+'''</table></div>''')